# KYC Qwen3.8-27B-FP8 — RAW OCR Diagnostic

**Objective:** select one customer folder, process every PDF and every actual PDF page, make exactly one Qwen OCR call per page, preserve the returned transcription as raw text, and measure the complete per-page pipeline timing.

This notebook deliberately excludes structured KYC extraction, database access, cross-document verification, duplicate collapsing, downstream validation, OCR fallback engines, and benchmark comparisons.

## 1. Configuration

Change only the configuration values needed for the experiment. `CUSTOMER_ID=None` enables reproducible random selection.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import Optional
import os

@dataclass
class Config:
    MODEL_PATH: str = "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.8-27B-FP8/main"

    # Set this to the directory containing customer folders.
    DATASET_ROOT: str = "/path/to/KYC_DATASET"

    CUSTOMER_ID: Optional[str] = None
    RANDOM_SEED: int = 42

    # High-resolution baseline. Change for controlled experiments.
    RENDER_DPI: int = 200

    # Controlled generation budget; suggested experiments: 128, 256, 512, 1024.
    MAX_NEW_TOKENS: int = 512

    MAX_QWEN_CALLS: int = 50

    WARMUP: bool = True
    WARMUP_MAX_NEW_TOKENS: int = 64

    # No fallback OCR is enabled in this diagnostic.
    ENABLE_OCR_FALLBACK: bool = False

    # Optional image preprocessing is disabled by default.
    ENABLE_OPTIONAL_PREPROCESSING: bool = False

    OUTPUT_DIR: str = "./raw_ocr_diagnostic_output"

    # Qwen/Transformers generation settings.
    DO_SAMPLE: bool = False
    TEMPERATURE: Optional[float] = None
    TOP_P: Optional[float] = None

    # Keep only PDFs for this experiment.
    PDF_EXTENSIONS = (".pdf",)

CFG = Config()
Path(CFG.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
print(f"MODEL_PATH       = {CFG.MODEL_PATH}")
print(f"DATASET_ROOT     = {CFG.DATASET_ROOT}")
print(f"CUSTOMER_ID      = {CFG.CUSTOMER_ID}")
print(f"RANDOM_SEED      = {CFG.RANDOM_SEED}")
print(f"RENDER_DPI       = {CFG.RENDER_DPI}")
print(f"MAX_NEW_TOKENS   = {CFG.MAX_NEW_TOKENS}")
print(f"MAX_QWEN_CALLS   = {CFG.MAX_QWEN_CALLS}")

## 2. Package installation

Run this cell only if the Domino environment needs the packages. It intentionally **does not install or import `flash_attn`**.

`flash-linear-attention==0.5.2` is retained because it is already part of the project environment; compatibility is diagnosed later and it is never force-enabled.

In [ ]:
# Required / project packages.
# Uncomment if your Domino image does not already contain them.
%pip install -q "pymupdf>=1.24" "transformers>=4.51" "accelerate>=1.5" "safetensors>=0.5" "pillow>=10" "pandas>=2" "packaging>=24" "flash-linear-attention==0.5.2"

# IMPORTANT:
# - Do NOT install flash_attn.
# - PyTorch is assumed to be supplied by the Domino/H100 environment.
# - If your approved Domino image pins compatible versions, prefer those pins over upgrading them here.

## 3. Imports

In [ ]:
import csv
import gc
import json
import random
import time
import traceback
import statistics
import platform
import sys
import importlib.util
from datetime import datetime, timezone
from pathlib import Path

import pymupdf
from PIL import Image
import pandas as pd
import torch
import transformers
from transformers import AutoProcessor, AutoModelForImageTextToText

try:
    from transformers import FineGrainedFP8Config
except ImportError:
    FineGrainedFP8Config = None

print("Imports complete.")

## 4. Environment diagnostics

In [ ]:
def package_version(name):
    try:
        mod = __import__(name)
        return getattr(mod, "__version__", "unknown")
    except Exception:
        return "not importable"

print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("pymupdf:", getattr(pymupdf, "version", None))
print("CUDA available:", torch.cuda.is_available())
print("CUDA runtime:", torch.version.cuda)
print("flash_attn importable:", importlib.util.find_spec("flash_attn") is not None)
print("flash_linear_attention importable:", importlib.util.find_spec("flash_linear_attention") is not None)
print("FineGrainedFP8Config available:", FineGrainedFP8Config is not None)

## 5. GPU diagnostics

In [ ]:
def gpu_snapshot():
    if not torch.cuda.is_available():
        return {
            "device": "cpu",
            "allocated_bytes": 0,
            "reserved_bytes": 0,
            "max_allocated_bytes": 0,
        }
    torch.cuda.synchronize()
    return {
        "device": torch.cuda.get_device_name(0),
        "allocated_bytes": torch.cuda.memory_allocated(0),
        "reserved_bytes": torch.cuda.memory_reserved(0),
        "max_allocated_bytes": torch.cuda.max_memory_allocated(0),
    }

print("GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {props.name}")
        print(f"  Total memory: {props.total_memory / 1024**3:.2f} GiB")
    snap = gpu_snapshot()
    print(f"Allocated: {snap['allocated_bytes']/1024**3:.2f} GiB")
    print(f"Reserved:  {snap['reserved_bytes']/1024**3:.2f} GiB")
else:
    print("WARNING: CUDA is unavailable. This notebook is intended for the Domino H100 environment.")

## 6. FP8 / model configuration

The loader first attempts to use the model's supported `FineGrainedFP8Config` through `quantization_config`. If the installed Transformers/model combination exposes a different required loading interface, the cell reports the exact error rather than silently changing the model.

No manual FP8 weight conversion is performed.

In [ ]:
MODEL = None
PROCESSOR = None
MODEL_DEVICE = None
QUANT_CONFIG = None

def build_quantization_config():
    if FineGrainedFP8Config is None:
        print("FineGrainedFP8Config is not available in this Transformers build.")
        return None

    # Inspect the constructor instead of guessing unsupported arguments.
    import inspect
    sig = inspect.signature(FineGrainedFP8Config)
    print("FineGrainedFP8Config signature:", sig)

    # Qwen3.8-27B-FP8 model repositories may already encode the FP8 configuration.
    # Prefer the model's own configuration when present; only construct a config if
    # the installed library exposes a zero-argument-compatible config.
    try:
        return FineGrainedFP8Config()
    except TypeError:
        print("FineGrainedFP8Config requires model-specific arguments; using the repository's encoded config.")
        return None

QUANT_CONFIG = build_quantization_config()
print("Explicit FineGrainedFP8Config object:", type(QUANT_CONFIG).__name__ if QUANT_CONFIG else None)

## 7. Attention backend diagnostics

`flash-linear-attention` is **not assumed to be compatible merely because it is installed**. This cell checks package presence and the Transformers model configuration. The model loader uses SDPA when supported by the installed Qwen/Transformers stack rather than forcing an unverified backend.

In [ ]:
def attention_diagnostics():
    print("flash_attn installed/importable:", importlib.util.find_spec("flash_attn") is not None)
    print("flash_linear_attention installed/importable:", importlib.util.find_spec("flash_linear_attention") is not None)
    print("Requested attention implementation for loader: sdpa")
    print("Reason: SDPA is a supported PyTorch/Transformers backend and avoids forcing an unverified flash-linear-attention integration.")

attention_diagnostics()

## 8. Load exactly one Qwen model and one processor

The model and processor are loaded once and reused for all pages.

In [ ]:
load_started = time.perf_counter()

model_kwargs = {
    "trust_remote_code": True,
    "device_map": "auto",
}

# Use SDPA when the installed Transformers/Qwen stack accepts it.
model_kwargs["attn_implementation"] = "sdpa"

if QUANT_CONFIG is not None:
    model_kwargs["quantization_config"] = QUANT_CONFIG

print("Loading model...")
print("Model path:", CFG.MODEL_PATH)

try:
    MODEL = AutoModelForImageTextToText.from_pretrained(
        CFG.MODEL_PATH,
        **model_kwargs,
    )
except TypeError as exc:
    # If this exact model build rejects an optional loader argument, retry only
    # after making the reason explicit. We never fall back to a different model.
    print("Model loader rejected an optional configuration:", repr(exc))
    print("Retrying with repository-native quantization configuration and no forced attention argument.")
    retry_kwargs = {
        "trust_remote_code": True,
        "device_map": "auto",
    }
    MODEL = AutoModelForImageTextToText.from_pretrained(
        CFG.MODEL_PATH,
        **retry_kwargs,
    )

MODEL.eval()

load_elapsed = time.perf_counter() - load_started
print(f"Model load time: {load_elapsed:.2f} s")

MODEL_DEVICE = next(MODEL.parameters()).device
print("Model device:", MODEL_DEVICE)
print("Model dtype sample:", next(MODEL.parameters()).dtype)
print("Model config quantization_config:", getattr(MODEL.config, "quantization_config", None))
print("Model config attn_implementation:", getattr(MODEL.config, "_attn_implementation", None))

if torch.cuda.is_available():
    snap = gpu_snapshot()
    print(f"Post-load allocated: {snap['allocated_bytes']/1024**3:.2f} GiB")
    print(f"Post-load reserved:  {snap['reserved_bytes']/1024**3:.2f} GiB")

In [ ]:
processor_started = time.perf_counter()
print("Loading processor...")
PROCESSOR = AutoProcessor.from_pretrained(
    CFG.MODEL_PATH,
    trust_remote_code=True,
)
print(f"Processor load time: {time.perf_counter() - processor_started:.2f} s")
print("Processor type:", type(PROCESSOR).__name__)

## 9. Raw OCR prompt

This is the only task sent to Qwen. The model receives the page image and must return only literal raw transcription.

In [ ]:
RAW_OCR_PROMPT = """
You are performing LITERAL VISUAL TRANSCRIPTION of a scanned Algerian administrative/KYC document page.

THE IMAGE IS THE ONLY SOURCE OF TRUTH.

Your task is to transcribe EVERYTHING that is visually readable on THIS PAGE, as raw text.

STRICT RULES:
- Transcribe visible text exactly as it appears.
- Do not guess.
- Do not complete partially visible words, names, numbers, or sentences.
- Do not correct spelling.
- Do not correct OCR-like ambiguities.
- Do not normalize accents or diacritics.
- Do not normalize whitespace or punctuation.
- Do not translate anything.
- Do not transliterate Arabic.
- Do not convert Arabic script into Latin script.
- Do not convert Latin script into Arabic script.
- Do not reorder names or words.
- Do not infer missing characters.
- Do not infer content from another page.
- Do not use information from another document.
- Do not use a database or external knowledge.
- Do not identify or extract predefined KYC fields.
- Do not classify the document.
- Do not interpret the meaning of the document.
- Do not reconstruct missing text.
- Do not silently omit a readable region because it is not a typical identity field.
- If text is genuinely unreadable, write [UNREADABLE] at that location instead of inventing content.
- Preserve the original visual reading order as closely as practical: normally top-to-bottom and left-to-right within a line, while respecting obvious columns/blocks.
- Preserve Arabic text in Arabic script.
- Preserve French text in French.
- Preserve Latin-script names exactly as visually written.
- Preserve numbers exactly as visually written.
- Preserve punctuation and special characters when visible.
- Preserve the literal '<' characters in machine-readable zones (MRZ) when they are visible.
- Attempt to transcribe printed text, handwritten text, stamps, headers, footers, labels, annotations, signatures or signature-area text, and other visible textual content. Do not invent a signature's content if it is not legible.
- Algerian documents may contain French, Arabic, Latin-script names, Arabic-script names, Algerian administrative terminology, wilayas, daïras, communes, identity documents, passports, handwritten information, stamps, photocopies, black-and-white scans, blur, compression, skew, uneven illumination, damaged scans, mixed printed/handwritten content, and MRZ. Handle these as visual transcription challenges, not as instructions to infer missing content.
- Do not use knowledge of standard Algerian document layouts to fill in text that cannot be seen.
- Do not output field names or a JSON object.
- Do not output markdown.
- Do not output explanations.
- Do not output confidence scores.
- Do not prefix the transcription with phrases such as "Transcription:".
- Return ONLY the raw transcription of the visible page.

If a visible region contains no readable text, do not invent text. If the page contains only unreadable material, return [UNREADABLE].
"""
print(RAW_OCR_PROMPT)

## 10. Dataset/customer discovery

In [ ]:
DATASET_ROOT = Path(CFG.DATASET_ROOT)

def discover_customer_folders(root: Path):
    if not root.exists():
        raise FileNotFoundError(f"DATASET_ROOT does not exist: {root}")
    if not root.is_dir():
        raise NotADirectoryError(f"DATASET_ROOT is not a directory: {root}")

    discovery_start = time.perf_counter()
    folders = sorted([p for p in root.iterdir() if p.is_dir()], key=lambda p: p.name.lower())
    elapsed = time.perf_counter() - discovery_start
    print(f"Customer discovery time: {elapsed:.3f} s")
    print(f"Customer folders found: {len(folders)}")
    return folders

CUSTOMER_FOLDERS = discover_customer_folders(DATASET_ROOT)
if not CUSTOMER_FOLDERS:
    raise RuntimeError("No customer folders were found under DATASET_ROOT.")

## 11. Reproducible random customer selection

In [ ]:
selection_rng = random.Random(CFG.RANDOM_SEED)

if CFG.CUSTOMER_ID is not None:
    matches = [p for p in CUSTOMER_FOLDERS if p.name == str(CFG.CUSTOMER_ID)]
    if not matches:
        raise FileNotFoundError(f"CUSTOMER_ID={CFG.CUSTOMER_ID!r} was not found under {DATASET_ROOT}")
    CUSTOMER_DIR = matches[0]
else:
    CUSTOMER_DIR = selection_rng.choice(CUSTOMER_FOLDERS)

SELECTED_CUSTOMER_ID = CUSTOMER_DIR.name

print(f"CUSTOMER SELECTED: {SELECTED_CUSTOMER_ID}")
print(f"CUSTOMER PATH: {CUSTOMER_DIR}")

## 12. PDF discovery

Every PDF in the selected customer folder is processed independently. No duplicate collapsing is performed.

In [ ]:
def discover_pdfs(customer_dir: Path):
    start = time.perf_counter()
    pdfs = sorted(
        [p for p in customer_dir.iterdir() if p.is_file() and p.suffix.lower() in CFG.PDF_EXTENSIONS],
        key=lambda p: p.name.lower()
    )
    elapsed = time.perf_counter() - start

    print(f"PDF discovery time: {elapsed:.3f} s")
    print("\nPDF FILES FOUND:")
    if not pdfs:
        print("  NONE")
    for i, pdf in enumerate(pdfs, 1):
        print(f"{i}. {pdf.name} | {pdf.stat().st_size / 1024:.1f} KiB")
    return pdfs

PDF_FILES = discover_pdfs(CUSTOMER_DIR)
if not PDF_FILES:
    raise RuntimeError(f"No PDF files found in selected customer folder: {CUSTOMER_DIR}")

## 13. Dynamic page inspection

The actual page count comes from `len(pdf)`. No document type is assigned a hardcoded page count.

In [ ]:
def inspect_pdfs(pdf_files):
    inspection = []
    for pdf_path in pdf_files:
        row = {
            "customer_id": SELECTED_CUSTOMER_ID,
            "pdf": pdf_path.name,
            "file_size_bytes": pdf_path.stat().st_size,
            "page_count": None,
            "open_time_s": None,
            "error": None,
            "pages": []
        }
        open_start = time.perf_counter()
        try:
            with pymupdf.open(pdf_path) as pdf:
                row["page_count"] = len(pdf)
                for page_index in range(len(pdf)):
                    page = pdf[page_index]
                    rect = page.rect
                    row["pages"].append({
                        "page": page_index + 1,
                        "width_pt": float(rect.width),
                        "height_pt": float(rect.height),
                    })
        except Exception as exc:
            row["error"] = f"{type(exc).__name__}: {exc}"
        row["open_time_s"] = time.perf_counter() - open_start
        inspection.append(row)

    print("\nDYNAMIC PAGE INSPECTION")
    print("=" * 60)
    total_pages = 0
    for row in inspection:
        print(f"{row['pdf']}")
        if row["error"]:
            print(f"  ERROR: {row['error']}")
            continue
        print(f"  Pages: {row['page_count']}")
        print(f"  Size:  {row['file_size_bytes'] / 1024:.1f} KiB")
        for p in row["pages"]:
            print(f"    Page {p['page']}: {p['width_pt']:.1f} x {p['height_pt']:.1f} pt")
        total_pages += row["page_count"]

    print(f"\nTOTAL ACTUAL PAGES: {total_pages}")
    return inspection, total_pages

PDF_INSPECTION, TOTAL_PAGES = inspect_pdfs(PDF_FILES)

if any(row["error"] for row in PDF_INSPECTION):
    print("\nWARNING: At least one PDF could not be opened. It will be reported explicitly and not silently ignored.")

PLANNED_QWEN_CALLS = sum((row["page_count"] or 0) for row in PDF_INSPECTION)

print("\n" + "=" * 50)
print("RAW OCR PLAN")
print("=" * 50)
print(f"Customer: {SELECTED_CUSTOMER_ID}")
print(f"PDFs: {len(PDF_FILES)}")
print(f"Pages: {PLANNED_QWEN_CALLS}")
print(f"Planned Qwen calls: {PLANNED_QWEN_CALLS}")
print(f"Maximum allowed calls: {CFG.MAX_QWEN_CALLS}")
print("=" * 50)

if PLANNED_QWEN_CALLS > CFG.MAX_QWEN_CALLS:
    raise RuntimeError(
        f"STOP BEFORE INFERENCE: planned_calls={PLANNED_QWEN_CALLS} "
        f"> CFG.MAX_QWEN_CALLS={CFG.MAX_QWEN_CALLS}. "
        "Increase the configured limit deliberately or select another customer."
    )

## 14. Page rendering

Each page is rendered exactly once at the configured DPI. Optional preprocessing is disabled by default.

In [ ]:
def render_page(page, dpi: int):
    render_start = time.perf_counter()
    matrix = pymupdf.Matrix(dpi / 72.0, dpi / 72.0)
    pix = page.get_pixmap(matrix=matrix, alpha=False)
    image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    render_time = time.perf_counter() - render_start
    return image, render_time

def optional_preprocess(image: Image.Image):
    # Intentionally a no-op baseline.
    # Any future preprocessing must be enabled explicitly and measured.
    return image

## 15. Qwen raw OCR inference

Exactly one generation is performed per document page. The function returns the literal decoded model output without field parsing or normalization.

In [ ]:
def build_qwen_messages(image: Image.Image):
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": RAW_OCR_PROMPT},
            ],
        }
    ]

def processor_prepare(image: Image.Image, max_new_tokens: int):
    prep_start = time.perf_counter()
    messages = build_qwen_messages(image)

    # Apply the processor's chat template when available. This is model-specific
    # and avoids manually constructing Qwen multimodal tokens.
    if hasattr(PROCESSOR, "apply_chat_template"):
        text = PROCESSOR.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        inputs = PROCESSOR(
            text=[text],
            images=[image],
            padding=True,
            return_tensors="pt",
        )
    else:
        inputs = PROCESSOR(
            text=[RAW_OCR_PROMPT],
            images=[image],
            padding=True,
            return_tensors="pt",
        )

    processor_time = time.perf_counter() - prep_start
    return inputs, processor_time

def move_inputs_to_model_device(inputs):
    transfer_start = time.perf_counter()
    target_device = MODEL_DEVICE
    moved = {}
    for key, value in inputs.items():
        moved[key] = value.to(target_device) if hasattr(value, "to") else value
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    transfer_time = time.perf_counter() - transfer_start
    return moved, transfer_time

def count_new_tokens(generated_ids, input_ids):
    try:
        if input_ids is not None and generated_ids.ndim == 2:
            prompt_len = input_ids.shape[-1]
            return max(0, int(generated_ids.shape[-1] - prompt_len))
    except Exception:
        pass
    try:
        return int(generated_ids.numel())
    except Exception:
        return None

def run_qwen_raw_ocr(image: Image.Image, max_new_tokens: int):
    inputs, processor_time = processor_prepare(image, max_new_tokens)
    moved_inputs, gpu_transfer_time = move_inputs_to_model_device(inputs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    generation_start = time.perf_counter()

    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": CFG.DO_SAMPLE,
    }
    if CFG.TEMPERATURE is not None:
        generation_kwargs["temperature"] = CFG.TEMPERATURE
    if CFG.TOP_P is not None:
        generation_kwargs["top_p"] = CFG.TOP_P

    with torch.inference_mode():
        generated = MODEL.generate(
            **moved_inputs,
            **generation_kwargs,
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    inference_time = time.perf_counter() - generation_start

    decode_start = time.perf_counter()
    input_ids = moved_inputs.get("input_ids")
    output_tokens = count_new_tokens(generated, input_ids)

    # Decode only the generated continuation. This prevents the prompt from being
    # accidentally written into the raw OCR output.
    if input_ids is not None and generated.ndim == 2:
        prompt_len = input_ids.shape[-1]
        continuation = generated[:, prompt_len:]
    else:
        continuation = generated

    raw_text = PROCESSOR.batch_decode(
        continuation,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]
    decode_time = time.perf_counter() - decode_start

    tokens_per_second = (
        output_tokens / inference_time
        if output_tokens is not None and inference_time > 0
        else None
    )

    return {
        "raw_text": raw_text,
        "processor_time_s": processor_time,
        "gpu_transfer_time_s": gpu_transfer_time,
        "inference_time_s": inference_time,
        "decode_time_s": decode_time,
        "input_tokens": int(input_ids.shape[-1]) if input_ids is not None else None,
        "output_tokens": output_tokens,
        "tokens_per_second": tokens_per_second,
    }

## 16. Timing helpers

CUDA synchronization is used around the actual GPU generation boundary so asynchronous CUDA work does not make generation appear artificially fast. Tiny CPU-only operations are not unnecessarily synchronized.

In [ ]:
def reset_peak_gpu_memory():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

def peak_gpu_memory_bytes():
    if torch.cuda.is_available():
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        return torch.cuda.max_memory_allocated()
    return 0

## 17. Optional warm-up

The warm-up is separate from document OCR and does not consume the page call budget. It is useful because model initialization and kernel compilation can affect the first real inference.

In [ ]:
WARMUP_RESULT = None

if CFG.WARMUP:
    print("=" * 60)
    print("OPTIONAL QWEN WARM-UP")
    print("=" * 60)

    # Use a tiny synthetic white page so no document content is involved.
    warmup_image = Image.new("RGB", (512, 512), "white")
    warmup_start = time.perf_counter()
    try:
        WARMUP_RESULT = run_qwen_raw_ocr(
            warmup_image,
            max_new_tokens=CFG.WARMUP_MAX_NEW_TOKENS,
        )
        WARMUP_RESULT["wall_time_s"] = time.perf_counter() - warmup_start
        print("Warm-up completed.")
        print(f"Warm-up wall time: {WARMUP_RESULT['wall_time_s']:.2f} s")
        print(f"Warm-up inference:  {WARMUP_RESULT['inference_time_s']:.2f} s")
        print(f"Warm-up output tokens: {WARMUP_RESULT['output_tokens']}")
    except Exception as exc:
        WARMUP_RESULT = {
            "status": "ERROR",
            "error": f"{type(exc).__name__}: {exc}",
            "wall_time_s": time.perf_counter() - warmup_start,
        }
        print("WARNING: warm-up failed.")
        print(WARMUP_RESULT["error"])
        print("The document diagnostic is not silently switched to another OCR engine.")
else:
    print("Warm-up disabled.")

## 18. RAW OCR execution

This is the core experiment:

**one PDF → one actual page → one rendered image → one Qwen call → one raw text result**

An error on a page is recorded and the next page continues when safe.

In [ ]:
raw_results = []
performance_rows = []
qwen_call_number = 0
experiment_start = time.perf_counter()

print("\n" + "=" * 70)
print("RAW OCR EXECUTION")
print("=" * 70)

for pdf_index, pdf_path in enumerate(PDF_FILES, 1):
    try:
        pdf_open_start = time.perf_counter()
        pdf = pymupdf.open(pdf_path)
        pdf_open_time = time.perf_counter() - pdf_open_start
    except Exception as exc:
        err = f"{type(exc).__name__}: {exc}"
        print(f"\n[PDF OPEN ERROR] {pdf_path.name}: {err}")
        raw_results.append({
            "customer_id": SELECTED_CUSTOMER_ID,
            "pdf": pdf_path.name,
            "page": None,
            "raw_text": "",
            "status": "ERROR",
            "stage": "pdf_open",
            "error": err,
        })
        performance_rows.append({
            "customer_id": SELECTED_CUSTOMER_ID,
            "pdf": pdf_path.name,
            "page": None,
            "render_time_s": 0.0,
            "processor_time_s": 0.0,
            "gpu_transfer_time_s": 0.0,
            "inference_time_s": 0.0,
            "decode_time_s": 0.0,
            "output_write_time_s": 0.0,
            "total_page_time_s": 0.0,
            "image_width": None,
            "image_height": None,
            "max_new_tokens": CFG.MAX_NEW_TOKENS,
            "input_tokens": None,
            "output_tokens": None,
            "tokens_per_second": None,
            "qwen_call_number": None,
            "status": "ERROR",
            "error": err,
        })
        continue

    try:
        actual_page_count = len(pdf)
        print(f"\nPDF {pdf_index}/{len(PDF_FILES)}: {pdf_path.name} | Pages: {actual_page_count}")

        for page_zero_index in range(actual_page_count):
            page_number = page_zero_index + 1
            qwen_call_number += 1

            page_total_start = time.perf_counter()
            page = None
            image = None
            result = None
            error = ""
            status = "OK"
            error_stage = None

            try:
                page = pdf[page_zero_index]
                image, render_time = render_page(page, CFG.RENDER_DPI)
                image = optional_preprocess(image)
                image_width, image_height = image.size

                print("\n" + "-" * 70)
                print(f"[QWEN {qwen_call_number}/{PLANNED_QWEN_CALLS}]")
                print(f"customer={SELECTED_CUSTOMER_ID}")
                print(f"pdf={pdf_path.name}")
                print(f"page={page_number}/{actual_page_count}")
                print(f"image={image_width}x{image_height}")
                print(f"max_new_tokens={CFG.MAX_NEW_TOKENS}")
                print(f"render_time={render_time:.3f} s")

                print("[QWEN START]")
                qwen_start_wall = time.perf_counter()
                reset_peak_gpu_memory()

                try:
                    result = run_qwen_raw_ocr(
                        image,
                        max_new_tokens=CFG.MAX_NEW_TOKENS,
                    )
                except Exception:
                    error_stage = "qwen_inference"
                    raise

                qwen_wall_elapsed = time.perf_counter() - qwen_start_wall

                if result["raw_text"] == "":
                    print("WARNING: Qwen returned empty OCR.")
                    print(f"prompt length (characters): {len(RAW_OCR_PROMPT)}")
                    print(f"input_tokens: {result['input_tokens']}")
                    print(f"generated token count: {result['output_tokens']}")
                    print(f"generation config: max_new_tokens={CFG.MAX_NEW_TOKENS}, do_sample={CFG.DO_SAMPLE}")
                    print(f"elapsed: {qwen_wall_elapsed:.3f} s")

                print("[QWEN END]")
                print(f"elapsed={result['inference_time_s']:.3f} s")
                print(f"output_tokens={result['output_tokens']}")
                print(f"tokens/sec={result['tokens_per_second']}")
                print("\nRAW OCR OUTPUT:")
                print(result["raw_text"])

            except Exception as exc:
                status = "ERROR"
                error = f"{type(exc).__name__}: {exc}"
                if error_stage is None:
                    error_stage = "page_processing"
                print(f"\nPAGE ERROR | stage={error_stage} | {error}")
                print(traceback.format_exc())

                image_width = image.width if image is not None else None
                image_height = image.height if image is not None else None
                render_time = locals().get("render_time", 0.0)

                result = {
                    "raw_text": "",
                    "processor_time_s": 0.0,
                    "gpu_transfer_time_s": 0.0,
                    "inference_time_s": 0.0,
                    "decode_time_s": 0.0,
                    "input_tokens": None,
                    "output_tokens": None,
                    "tokens_per_second": None,
                }

            total_page_time = time.perf_counter() - page_total_start
            peak_mem = peak_gpu_memory_bytes()

            raw_results.append({
                "customer_id": SELECTED_CUSTOMER_ID,
                "pdf": pdf_path.name,
                "page": page_number,
                "image_width": image_width,
                "image_height": image_height,
                "raw_text": result["raw_text"],
                "qwen_call": qwen_call_number,
                "max_new_tokens": CFG.MAX_NEW_TOKENS,
                "inference_time_s": result["inference_time_s"],
                "status": status,
                "error": error,
            })

            performance_rows.append({
                "customer_id": SELECTED_CUSTOMER_ID,
                "pdf": pdf_path.name,
                "page": page_number,
                "render_time_s": render_time,
                "processor_time_s": result["processor_time_s"],
                "gpu_transfer_time_s": result["gpu_transfer_time_s"],
                "inference_time_s": result["inference_time_s"],
                "decode_time_s": result["decode_time_s"],
                "output_write_time_s": 0.0,  # populated by the writer timing cell
                "total_page_time_s": total_page_time,
                "image_width": image_width,
                "image_height": image_height,
                "max_new_tokens": CFG.MAX_NEW_TOKENS,
                "input_tokens": result["input_tokens"],
                "output_tokens": result["output_tokens"],
                "tokens_per_second": result["tokens_per_second"],
                "qwen_call_number": qwen_call_number,
                "status": status,
                "error": error,
                "peak_gpu_memory_bytes": peak_mem,
            })

    finally:
        pdf.close()

experiment_elapsed = time.perf_counter() - experiment_start
print("\n" + "=" * 70)
print(f"RAW OCR EXECUTION COMPLETE: {experiment_elapsed:.2f} s")
print("=" * 70)

## 19. Raw result writers

The raw model output is not parsed, normalized, corrected, translated, or structured. JSONL/CSV metadata surrounds the untouched `raw_text`.

In [ ]:
OUTPUT_DIR = Path(CFG.OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

JSONL_PATH = OUTPUT_DIR / "raw_ocr_results.jsonl"
TXT_PATH = OUTPUT_DIR / "raw_ocr_results.txt"
CSV_PATH = OUTPUT_DIR / "raw_ocr_results.csv"
PERF_PATH = OUTPUT_DIR / "raw_ocr_performance.csv"

def write_raw_outputs(raw_rows, perf_rows):
    write_start = time.perf_counter()

    # JSONL: metadata is structured; raw_text is untouched.
    with JSONL_PATH.open("w", encoding="utf-8") as f:
        for row in raw_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    # Human-readable TXT.
    with TXT_PATH.open("w", encoding="utf-8") as f:
        for row in raw_rows:
            f.write("=" * 70 + "\n")
            f.write(f"CUSTOMER: {row.get('customer_id', '')}\n")
            f.write(f"PDF: {row.get('pdf', '')}\n")
            f.write(f"PAGE: {row.get('page', '')}\n")
            if row.get("image_width") is not None:
                f.write(f"IMAGE: {row['image_width']} x {row['image_height']}\n")
            f.write("=" * 70 + "\n")
            f.write(row.get("raw_text", ""))
            f.write("\n\n")

    # Metadata/performance CSV.
    pd.DataFrame(perf_rows).to_csv(CSV_PATH, index=False, encoding="utf-8-sig")

    output_write_time = time.perf_counter() - write_start
    return output_write_time

output_write_time = write_raw_outputs(raw_results, performance_rows)

# The requested output_write_time is also represented in the performance CSV.
# This write is one batch operation after all page OCR, so it is not falsely
# attributed as a per-page filesystem measurement.
for row in performance_rows:
    row["output_write_time_s"] = output_write_time / max(len(performance_rows), 1)

pd.DataFrame(performance_rows).to_csv(PERF_PATH, index=False, encoding="utf-8-sig")

print("Outputs written:")
print(JSONL_PATH)
print(TXT_PATH)
print(CSV_PATH)
print(PERF_PATH)
print(f"Batch output-writing time: {output_write_time:.3f} s")

## 20. Performance report

In [ ]:
perf_df = pd.DataFrame(performance_rows)

if not perf_df.empty:
    ok_df = perf_df[perf_df["status"] == "OK"].copy()

    def safe_mean(series):
        s = pd.to_numeric(series, errors="coerce").dropna()
        return float(s.mean()) if len(s) else float("nan")

    def safe_median(series):
        s = pd.to_numeric(series, errors="coerce").dropna()
        return float(s.median()) if len(s) else float("nan")

    def safe_p95(series):
        s = pd.to_numeric(series, errors="coerce").dropna()
        return float(s.quantile(0.95)) if len(s) else float("nan")

    print("=" * 70)
    print("RAW OCR DIAGNOSTIC SUMMARY")
    print("=" * 70)
    print(f"Customer: {SELECTED_CUSTOMER_ID}")
    print(f"PDFs: {len(PDF_FILES)}")
    print(f"Pages planned: {PLANNED_QWEN_CALLS}")
    print(f"Qwen calls attempted: {qwen_call_number}")
    print()
    print(f"Total runtime: {experiment_elapsed:.3f} s")
    print(f"Total rendering: {safe_mean(perf_df['render_time_s']) * len(perf_df):.3f} s (sum)")
    print(f"Total preprocessing: 0.000 s (baseline no-op)")
    print(f"Total processor time: {pd.to_numeric(perf_df['processor_time_s'], errors='coerce').sum():.3f} s")
    print(f"Total GPU transfer: {pd.to_numeric(perf_df['gpu_transfer_time_s'], errors='coerce').sum():.3f} s")
    print(f"Total inference: {pd.to_numeric(perf_df['inference_time_s'], errors='coerce').sum():.3f} s")
    print(f"Total decoding: {pd.to_numeric(perf_df['decode_time_s'], errors='coerce').sum():.3f} s")
    print(f"Total output writing: {output_write_time:.3f} s")
    print()
    print(f"Average Qwen latency: {safe_mean(ok_df['inference_time_s']):.3f} s")
    print(f"Median Qwen latency: {safe_median(ok_df['inference_time_s']):.3f} s")
    print(f"P95 Qwen latency: {safe_p95(ok_df['inference_time_s']):.3f} s")
    print()
    print(f"Average output tokens: {safe_mean(ok_df['output_tokens']):.1f}")
    print(f"Average tokens/sec: {safe_mean(ok_df['tokens_per_second']):.2f}")
    print("=" * 70)

    print("\nPer-page performance:")
    display(perf_df)
else:
    print("No performance rows were produced.")

## 21. Final diagnostic summary and files

In [ ]:
successful_pages = sum(r.get("status") == "OK" for r in raw_results)
failed_pages = sum(r.get("status") == "ERROR" for r in raw_results)
empty_pages = sum(
    r.get("status") == "OK" and r.get("raw_text", "") == ""
    for r in raw_results
)

print("=" * 70)
print("FINAL RAW OCR DIAGNOSTIC")
print("=" * 70)
print(f"Customer selected:       {SELECTED_CUSTOMER_ID}")
print(f"PDFs discovered:         {len(PDF_FILES)}")
print(f"Actual pages discovered: {PLANNED_QWEN_CALLS}")
print(f"Qwen calls attempted:    {qwen_call_number}")
print(f"Successful page calls:   {successful_pages}")
print(f"Failed page calls:       {failed_pages}")
print(f"Empty OCR responses:     {empty_pages}")
print(f"Render DPI:              {CFG.RENDER_DPI}")
print(f"Max new tokens:          {CFG.MAX_NEW_TOKENS}")
print()
print("RAW OUTPUT FILES")
print(f"JSONL: {JSONL_PATH.resolve()}")
print(f"TXT:   {TXT_PATH.resolve()}")
print(f"CSV:   {CSV_PATH.resolve()}")
print(f"PERF:  {PERF_PATH.resolve()}")
print("=" * 70)

if failed_pages:
    print("\nERRORS:")
    for row in raw_results:
        if row.get("status") == "ERROR":
            print(f"- {row.get('pdf')} page {row.get('page')}: {row.get('error')}")

print("\nThis notebook intentionally produces no KYC status, field extraction, MRZ validation, database comparison,")
print("duplicate collapsing, cross-document verification, OCR fallback, or normalization.")
print("The raw text above/files are the direct diagnostic evidence of what Qwen returned per page.")